# Choose a 4-bit format using fewer input examples

We want a weight format that keeps the multiplication result close to `W @ X_cal`. This notebook first compares formats using a small sample of calibration examples, then checks the choices against full-output evaluations.

The fast method samples input examples—columns of `X_cal`. It still quantizes every weight for each format.


## The five number formats

The candidates are E2M1, E1M2, MXUINT4, MXINT4, and BFP4. Each format rounds weights to a small set of allowed values.

Normally, groups of up to 32 consecutive weights in each row share a scale. The scale sets the spacing between allowed values. Rounding uses nearest values with ties-to-even, and values outside the allowed range are clipped.

MXUINT4 uses codes 0–15 and subtracts 8 when decoding. This custom convention allows negative reconstructed weights too. Saved matrices contain the rounded floating-point values, rather than packed 4-bit data.


## 1. How the algorithms work

The next cell defines the rounding functions, fast selector, scale search, and comparison functions.

**Fast selection**

1. If there are at most 32 calibration examples, use them all.
2. Otherwise, square and sum each input column to measure its energy. Keep the column with the largest energy and select 31 other columns randomly, without duplicates. A fixed seed makes this selection repeatable.
3. Quantize the complete weight matrix with each format using its baseline scales. Multiply the weight errors by the selected input columns to find output errors.
4. Estimate the average squared output error over all calibration examples. The largest-energy example counts once. The 31 randomly sampled examples represent all the remaining examples, so their total squared error is multiplied by `(number_of_examples − 1) / 31`. Take the square root to obtain the estimated RMSE. Lower is better.
5. Choose the best format. If leading scores are almost equal and only a sample was used, check the tied formats on all calibration examples when the weights are not all zero. Any remaining tie uses the fixed format order.

**Example:** with 63 calibration examples, keep the largest-energy column and 31 of the remaining 62 columns. If the first column's squared output error sums to 10 and the 31 random columns' errors sum to 20, estimate the total as `10 + (62 / 31) × 20 = 50`. Divide by the number of output rows times 63, then take the square root. Every format uses the same sampled columns.

This saves output-scoring work, but does not guarantee the same winner as checking every example. All candidate weight matrices are still fully quantized.

**Scale-search reference**

The reference tries combinations of nearby block exponents and measures the complete calibration-output error. An exponent controls a power-of-two scale. Checking blocks together accounts for their errors cancelling or reinforcing each other.

The search starts within the configured range, usually ±3 around baseline exponents, and can widen up to ±5. In `exact` mode, it enumerates candidate combinations within the size limit. In `auto` mode, large searches can instead use a heuristic: start from several choices and improve block choices iteratively. The reported `EXACT` label means exhaustive search within the candidate range; a heuristic result does not guarantee the best combination.

The full-baseline comparison keeps baseline scales and checks every calibration example. This separates the effect of sampling from the effect of optimizing scales.


In [1]:
"""Define the format quantizers, sampled selector, and scale-search comparison."""

from __future__ import annotations

import argparse
import math
import time
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np

FORMATS = ["E2M1", "E1M2", "MXUINT4", "MXINT4", "BFP4"]
E8M0_MIN_EXP = -127
E8M0_MAX_EXP = 127
BLOCK_SIZE = 32
BFP_EXP_POLICY = "autoflex_predict"
BFP_ALPHA = 1.0

FORMAT_CONFIGS = {
    "E2M1": {
        "kind": "mxfp", "block_size": 32, "element_bits": 4,
        "metadata_bits": 8, "layout": "[S|E|M] = [1|2|1]",
        "grid": np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0]),
        "baseline_exp_shift": -2, "exp_min": -127, "exp_max": 127,
        "shared_scale": "power-of-two 2^e; 8-bit E8M0-style shared exponent",
        "convention_status": "classroom/OCP-MX convention used in this project",
    },
    "E1M2": {
        "kind": "mxfp", "block_size": 32, "element_bits": 4,
        "metadata_bits": 8, "layout": "[S|E|M] = [1|1|2]",
        "grid": np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5]),
        "baseline_exp_shift": -1, "exp_min": -127, "exp_max": 127,
        "shared_scale": "power-of-two 2^e; 8-bit E8M0-style shared exponent",
        "convention_status": "adopted classroom-consistent E1M2 convention",
    },
    "MXINT4": {
        "kind": "mxint", "block_size": 32, "element_bits": 4,
        "metadata_bits": 8, "layout": "[S|I|F] = [1|1|2]",
        "qmin": -8, "qmax": 7, "element_scale": 0.25,
        "exp_min": -127, "exp_max": 127,
        "shared_scale": "power-of-two 2^e; element precision scale 2^-2",
        "convention_status": "adopted from the classroom MXINT construction",
    },
    "BFP4": {
        "kind": "bfp", "block_size": 32, "element_bits": 4,
        "metadata_bits": 8,
        "layout": "flex4+8: 4-bit two's-complement integer mantissa (-8..7) + shared exponent",
        "mantissa_bits": 4, "qmin": -8, "qmax": 7,
        "exp_min": -127, "exp_max": 127,
        "shared_scale": "kappa = 2^E; 8-bit shared exponent per group",
        "convention_status": "Flexpoint-style convention; Autoflex exponent rule; row-wise block sharing",
    },
    "MXUINT4": {
        "kind": "mxuint", "block_size": 32, "element_bits": 4,
        "metadata_bits": 8,
        "layout": "unsigned 4-bit code; fixed midpoint zero point q_u - 8",
        "exp_min": -127, "exp_max": 127,
        "shared_scale": "power-of-two 2^e; fixed zero point 8; no per-block offset",
        "convention_status": "custom midpoint-coded unsigned-storage convention",
    },
}

def validate_matrices(W: np.ndarray, X: np.ndarray, x_name: str = "X") -> Tuple[np.ndarray, np.ndarray]:
    """Validate W and X and return float64 views/copies for analysis."""
    if np.iscomplexobj(W) or np.iscomplexobj(X):
        raise ValueError("W and X must contain real numbers, not complex numbers.")
    W = np.asarray(W, dtype=np.float64)
    X = np.asarray(X, dtype=np.float64)
    if W.ndim != 2 or X.ndim != 2 or W.size == 0 or X.size == 0:
        raise ValueError("W and X must be nonempty two-dimensional matrices.")
    if W.shape[1] != X.shape[0]:
        raise ValueError(
            f"Incompatible shapes: W={W.shape}, {x_name}={X.shape}; "
            "need W.shape[1] == X.shape[0]."
        )
    if not np.all(np.isfinite(W)) or not np.all(np.isfinite(X)):
        raise ValueError("W and X must contain only finite values (no NaN or Inf).")
    return W, X

def nearest_grid_rne(values: np.ndarray, grid: np.ndarray) -> np.ndarray:
    """Nearest codebook level; exact-distance ties prefer an even grid index."""
    values = np.asarray(values, dtype=np.float64)
    out = np.empty_like(values)
    flat_in = values.ravel()
    flat_out = out.ravel()
    for i, value in enumerate(flat_in):
        distances = np.abs(grid - value)
        dmin = float(np.min(distances))
        choices = np.flatnonzero(np.isclose(distances, dmin, rtol=0.0, atol=1e-15))
        if len(choices) == 1:
            index = int(choices[0])
        else:
            even_choices = choices[choices % 2 == 0]
            index = int(even_choices[0] if len(even_choices) else choices[0])
        flat_out[i] = grid[index]
    return out

def bfp_exp_autoflex_predict(block: np.ndarray) -> int:
    """Flexpoint/Autoflex predicted exponent for a static block, alpha=1."""
    maximum = float(np.max(np.abs(block)))
    if maximum == 0.0:
        return 0

    return int(np.ceil(np.log2(BFP_ALPHA * maximum))) - 3

def baseline_exp(block: np.ndarray, fmt: str) -> int:
    """Notebook's baseline scale selection for one block."""
    block = np.asarray(block, dtype=np.float64)
    maximum = float(np.max(np.abs(block)))
    if maximum == 0.0:
        return 0
    cfg = FORMAT_CONFIGS[fmt]
    raw = int(np.floor(np.log2(maximum)))
    if cfg["kind"] == "mxfp":
        exponent = raw + int(cfg["baseline_exp_shift"])
    elif cfg["kind"] == "mxuint":
        neg_extent = max(0.0, -float(np.min(block)))
        pos_extent = max(0.0, float(np.max(block)))
        required_step = max(neg_extent / 8.0, pos_extent / 7.0)
        exponent = int(np.ceil(np.log2(required_step))) if required_step > 0 else 0
    elif cfg["kind"] == "bfp":
        if BFP_EXP_POLICY != "autoflex_predict":
            raise ValueError(f"Unsupported BFP_EXP_POLICY={BFP_EXP_POLICY!r}")
        exponent = bfp_exp_autoflex_predict(block)
    else:
        exponent = raw
    return int(np.clip(exponent, cfg.get("exp_min", E8M0_MIN_EXP), cfg.get("exp_max", E8M0_MAX_EXP)))

def quantize_block_at_exp(block: np.ndarray, fmt: str, exponent: int) -> np.ndarray:
    """Quantize one block at an explicitly supplied shared exponent."""
    block = np.asarray(block, dtype=np.float64)
    cfg = FORMAT_CONFIGS[fmt]
    exponent = int(np.clip(exponent, cfg.get("exp_min", E8M0_MIN_EXP), cfg.get("exp_max", E8M0_MAX_EXP)))

    if cfg["kind"] == "mxfp":
        scale = np.exp2(float(exponent))
        local = np.abs(block) / scale
        rounded = nearest_grid_rne(local, cfg["grid"])
        return np.sign(block) * rounded * scale
    if cfg["kind"] == "mxint":
        step = np.exp2(float(exponent)) * float(cfg["element_scale"])
        q = np.clip(np.rint(block / step), cfg["qmin"], cfg["qmax"])
        return q * step
    if cfg["kind"] == "bfp":
        scale = np.exp2(float(exponent))
        mantissa = np.clip(np.rint(block / scale), cfg["qmin"], cfg["qmax"])
        return mantissa * scale
    if cfg["kind"] == "mxuint":
        scale = np.exp2(float(exponent))
        unsigned_code = np.clip(np.rint(block / scale) + 8, 0, 15)
        signed_code = unsigned_code - 8
        return signed_code * scale
    raise ValueError(f"Unknown format: {fmt}")

def rowwise_ranges(cols: int, block_size: int = BLOCK_SIZE) -> Iterable[Tuple[int, int]]:
    for start in range(0, cols, block_size):
        yield start, min(start + block_size, cols)

def quantize_matrix_baseline(W: np.ndarray, fmt: str, bfp_sharing: str = "block") -> np.ndarray:
    """Apply the brute-force notebook's default fixed-scale quantizer."""
    W = np.asarray(W, dtype=np.float64)
    if fmt == "BFP4" and bfp_sharing == "tensor":
        exponent = baseline_exp(W.ravel(), fmt)
        return quantize_block_at_exp(W, fmt, exponent)
    Wq = np.empty_like(W)
    block_size = FORMAT_CONFIGS[fmt]["block_size"]
    for row in range(W.shape[0]):
        for start, end in rowwise_ranges(W.shape[1], block_size):
            block = W[row, start:end]
            Wq[row, start:end] = quantize_block_at_exp(block, fmt, baseline_exp(block, fmt))
    return Wq

def best_group(scores: Dict[str, float]) -> List[str]:
    """Return all near-best candidates in the fixed order used by the notebook."""
    lowest = min(scores.values())
    group = []
    for name in FORMATS:
        if name in scores:
            value = scores[name]
            tolerance = 1e-15 + 1e-6 * max(abs(lowest), abs(value))
            if abs(value - lowest) <= tolerance:
                group.append(name)
    return group

# Use the same sampled input examples for every format.
def select_format(W: np.ndarray, X: np.ndarray, seed: int = 271828,
                  bfp_sharing: str = "block") -> dict:
    """Fast selector: score baseline quantizers on 1 high-energy + 31 random X columns."""
    W, X = validate_matrices(W, X)
    total_columns = X.shape[1]
    if total_columns <= 32:
        columns = np.arange(total_columns)
    else:
        energies = np.sum(X * X, axis=0)
        largest = int(np.argmax(energies))
        remaining = np.delete(np.arange(total_columns), largest)
        rng = np.random.default_rng(seed)
        random_columns = rng.permutation(remaining)[:31]
        columns = np.concatenate(([largest], random_columns))
    sample_X = X[:, columns]

    matrices: Dict[str, np.ndarray] = {}
    scores: Dict[str, float] = {}
    differences: Dict[str, np.ndarray] = {}
    for name in FORMATS:
        matrices[name] = quantize_matrix_baseline(W, name, bfp_sharing=bfp_sharing)
        differences[name] = matrices[name] - W
        error = differences[name] @ sample_X
        if total_columns <= 32:
            mse = float(np.mean(error * error))
        else:
            column_errors = np.sum(error * error, axis=0)
            tail_weight = (total_columns - 1) / 31.0
            estimated_total = column_errors[0] + tail_weight * np.sum(column_errors[1:])
            mse = float(estimated_total / (W.shape[0] * total_columns))
        scores[name] = float(np.sqrt(mse))

    finalists = best_group(scores)
    full_tie_scores: Dict[str, float] = {}
    if len(finalists) > 1 and total_columns > 32 and np.any(W != 0):
        for name in finalists:
            err = differences[name] @ X
            full_tie_scores[name] = float(np.sqrt(np.mean(err * err)))
        chosen = best_group(full_tie_scores)[0]
    else:
        chosen = finalists[0]

    return {
        "chosen": chosen,
        "W_quantized": matrices[chosen],
        "selected_columns": columns.tolist(),
        "sampled_scores": scores,
        "full_tie_scores": full_tie_scores,
        "quantized_by_format": matrices,
    }

def output_error_pct(W: np.ndarray, Wq: np.ndarray, X: np.ndarray,
                     reference_output: Optional[np.ndarray] = None) -> float:
    """100 * ||W X - Wq X||_F / ||W X||_F, matching notebook metric."""
    reference = W @ X if reference_output is None else reference_output
    error = (W - Wq) @ X
    denominator = float(np.linalg.norm(reference, ord="fro"))
    numerator = float(np.linalg.norm(error, ord="fro"))
    if denominator == 0.0:
        return 0.0 if numerator == 0.0 else float("inf")
    return 100.0 * numerator / denominator

def _gram_factor(X: np.ndarray) -> np.ndarray:
    """Return F such that F F^T = X X^T on its nonzero eigenspace."""
    gram = X @ X.T
    eigenvalues, eigenvectors = np.linalg.eigh(gram)
    if eigenvalues.size == 0 or float(np.max(eigenvalues)) <= 0.0:
        return np.zeros((X.shape[0], 0), dtype=np.float64)
    keep = eigenvalues > float(np.max(eigenvalues)) * 1e-14
    return eigenvectors[:, keep] * np.sqrt(eigenvalues[keep])

def _enumerate_exact(D: Sequence[np.ndarray]) -> Tuple[List[int], float]:
    """Find the exponent-choice combination minimizing squared norm of summed residuals."""
    sums = D[0]
    for d in D[1:]:
        sums = (sums[:, None, :] + d[None, :, :]).reshape(-1, sums.shape[-1])
    sse = np.einsum("ij,ij->i", sums, sums)
    best_flat = int(np.argmin(sse))
    shape = [len(d) for d in D]
    indices = [int(i) for i in np.unravel_index(best_flat, shape)]
    return indices, float(sse[best_flat])

def _reference_heuristic(D: Sequence[np.ndarray], rng: np.random.Generator,
                         n_random: int, pair_moves: bool) -> Tuple[List[int], float]:
    """Multi-start coordinate descent; a feasible solution, not a global proof."""
    n_blocks = len(D)
    counts = [len(d) for d in D]
    sq = [np.einsum("ij,ij->i", d, d) for d in D]
    starts = [[0] * n_blocks, [int(np.argmin(s)) for s in sq]]
    starts += [[int(rng.integers(k)) for k in counts] for _ in range(n_random)]
    best_idx: Optional[List[int]] = None
    best_value = np.inf

    for start in starts:
        idx = list(start)
        residual = sum(D[b][idx[b]] for b in range(n_blocks))
        current = float(residual @ residual)
        improved = True
        while improved:
            improved = False
            for b in range(n_blocks):
                base = residual - D[b][idx[b]]
                candidates = base[None, :] + D[b]
                values = np.einsum("ij,ij->i", candidates, candidates)
                k = int(np.argmin(values))
                if values[k] < current * (1.0 - 1e-12):
                    idx[b] = k
                    residual = candidates[k].copy()
                    current = float(values[k])
                    improved = True
            if not improved and pair_moves:
                for b in range(n_blocks):
                    for c in range(b + 1, n_blocks):
                        base = residual - D[b][idx[b]] - D[c][idx[c]]
                        candidates = (base[None, None, :] + D[b][:, None, :] + D[c][None, :, :])
                        values = np.einsum("ijk,ijk->ij", candidates, candidates)
                        i, j = np.unravel_index(int(np.argmin(values)), values.shape)
                        if values[i, j] < current * (1.0 - 1e-12):
                            idx[b], idx[c] = int(i), int(j)
                            residual = candidates[i, j].copy()
                            current = float(values[i, j])
                            improved = True
        if current < best_value:
            best_value = current
            best_idx = list(idx)

    assert best_idx is not None
    return best_idx, best_value

def _reference_candidates(W: np.ndarray, fmt: str, row: int, block_index: int,
                          start: int, end: int, window: int,
                          cache: dict) -> Tuple[int, np.ndarray, np.ndarray]:
    block = W[row, start:end]
    base = baseline_exp(block, fmt)
    cfg = FORMAT_CONFIGS[fmt]
    exponents: List[int] = []
    for offset in sorted(range(-window, window + 1), key=lambda off: (abs(off), off)):
        exponent = int(np.clip(base + offset, cfg["exp_min"], cfg["exp_max"]))
        if exponent not in exponents:
            exponents.append(exponent)
    candidates = []
    for exponent in exponents:
        key = (fmt, row, block_index, exponent)
        if key not in cache:
            cache[key] = quantize_block_at_exp(block, fmt, exponent)
        candidates.append(cache[key])
    return base, np.asarray(exponents, dtype=int), np.asarray(candidates, dtype=np.float64)

def _bfp_tensor_search(W: np.ndarray, X: np.ndarray, ref_window: int,
                       ref_window_max: int) -> Tuple[np.ndarray, dict]:
    cfg = FORMAT_CONFIGS["BFP4"]
    base = baseline_exp(W.ravel(), "BFP4")
    window = ref_window
    started = time.perf_counter()
    while True:
        exponents = sorted(
            {int(np.clip(base + off, cfg["exp_min"], cfg["exp_max"])) for off in range(-window, window + 1)},
            key=lambda exponent: (abs(exponent - base), exponent - base),
        )
        best = None
        for exponent in exponents:
            Wq = quantize_block_at_exp(W, "BFP4", exponent)
            residual = (W - Wq) @ X
            sse = float(np.sum(residual * residual))
            if best is None or sse < best[0]:
                best = (sse, exponent, Wq)
        assert best is not None
        edge = abs(best[1] - base) == window
        if edge and window < ref_window_max:
            window += 1
            continue
        break
    return best[2], {
        "exact_rows": W.shape[0], "heuristic_rows": 0,
        "widened_rows": W.shape[0] if window > ref_window else 0,
        "edge_rows": W.shape[0] if edge else 0,
        "offsets": [int(best[1] - base)],
        "seconds": time.perf_counter() - started,
        "method": "EXACT (tensor-wide BFP exponent)",
    }

# Search block scales against the complete calibration output.
def strong_reference_search(W: np.ndarray, X: np.ndarray, fmt: str,
                            mode: str = "auto", ref_window: int = 3,
                            ref_window_max: int = 5, ref_max_elems: int = 2_000_000,
                            n_random: int = 4, pair_moves: bool = True,
                            seed: int = 0, bfp_sharing: str = "block") -> Tuple[np.ndarray, dict]:
    """Search shared-exponent offsets to minimize actual calibration-output SSE.

    mode='exact' forces exhaustive enumeration and errors when the configured
    cap would be exceeded. mode='auto' enumerates exactly when feasible and
    otherwise uses documented multi-start coordinate descent. mode='heuristic'
    always uses coordinate descent. Search starts at +/-ref_window and widens
    to +/-ref_window_max when the best candidate hits a window edge.
    """
    W, X = validate_matrices(W, X)
    if fmt not in FORMATS:
        raise ValueError(f"Unknown format {fmt!r}; choose from {FORMATS}.")
    if mode not in {"auto", "exact", "heuristic"}:
        raise ValueError("mode must be one of: 'auto', 'exact', 'heuristic'.")
    if ref_window < 0 or ref_window_max < ref_window:
        raise ValueError("Require 0 <= ref_window <= ref_window_max.")
    if ref_max_elems < 1:
        raise ValueError("ref_max_elems must be at least 1.")
    if fmt == "BFP4" and bfp_sharing == "tensor":
        return _bfp_tensor_search(W, X, ref_window, ref_window_max)

    factor = _gram_factor(X)
    block_size = FORMAT_CONFIGS[fmt]["block_size"]
    ranges = list(rowwise_ranges(W.shape[1], block_size))
    n_blocks = len(ranges)
    feature_dim = max(1, factor.shape[1])
    limit = max(1, ref_max_elems // feature_dim)
    Wq = np.zeros_like(W)
    info = {"exact_rows": 0, "heuristic_rows": 0, "widened_rows": 0,
            "edge_rows": 0, "offsets": []}
    cache: dict = {}
    started = time.perf_counter()

    for row in range(W.shape[0]):
        terms = [W[row, start:end] @ factor[start:end] for start, end in ranges]
        rng = np.random.default_rng(seed + row)
        window = ref_window
        while True:
            candidates = [
                _reference_candidates(W, fmt, row, b, start, end, window, cache)
                for b, (start, end) in enumerate(ranges)
            ]
            residual_options = [
                terms[b] - candidates[b][2] @ factor[start:end]
                for b, (start, end) in enumerate(ranges)
            ]
            combinations = math.prod(len(d) for d in residual_options)
            if mode == "exact":
                exact = True
            elif mode == "heuristic":
                exact = False
            else:
                exact = combinations <= limit
            if exact and combinations > limit:
                raise RuntimeError(
                    f"{fmt} row {row}: {combinations:,} exponent combinations exceed "
                    f"the exact-search cap {limit:,}; increase --max-search-elements "
                    "or use --reference-mode auto/heuristic."
                )
            if exact:
                indices, _ = _enumerate_exact(residual_options)
            else:
                indices, _ = _reference_heuristic(residual_options, rng, n_random, pair_moves)

            offsets = [int(candidates[b][1][indices[b]] - candidates[b][0]) for b in range(n_blocks)]
            edge = any(abs(offset) == window for offset in offsets)
            can_widen_exact = ((2 * (window + 1) + 1) ** n_blocks) <= limit
            if edge and window < ref_window_max and (not exact or can_widen_exact):
                window += 1
                continue
            break

        info["exact_rows"] += int(exact)
        info["heuristic_rows"] += int(not exact)
        info["widened_rows"] += int(window > ref_window)
        info["edge_rows"] += int(edge)
        info["offsets"].extend(offsets)
        for b, (start, end) in enumerate(ranges):
            Wq[row, start:end] = candidates[b][2][indices[b]]

    info["seconds"] = time.perf_counter() - started
    if info["heuristic_rows"] == 0:
        info["method"] = "EXACT"
    elif info["exact_rows"] == 0:
        info["method"] = "HEURISTIC (feasible solution; not global proof)"
    else:
        info["method"] = "MIXED exact/heuristic"
    info["search_cap_combinations"] = int(limit)
    info["window_initial"] = int(ref_window)
    info["window_max"] = int(ref_window_max)
    return Wq, info

# Compare sampled selection with full-baseline and scale-search results.
def compare_algorithms(W: np.ndarray, X: np.ndarray, seed: int = 271828,
                       reference_mode: str = "auto", ref_window: int = 3,
                       ref_window_max: int = 5, ref_max_elems: int = 2_000_000,
                       n_random: int = 4, pair_moves: bool = True,
                       bfp_sharing: str = "block", run_reference: bool = True) -> dict:
    """Run fast selection and, optionally, the scale-optimized reference for all formats."""
    W, X = validate_matrices(W, X)
    fast_start = time.perf_counter()
    fast = select_format(W, X, seed=seed, bfp_sharing=bfp_sharing)
    fast_seconds = time.perf_counter() - fast_start

    reference_output = W @ X
    baseline_errors = {
        fmt: output_error_pct(W, fast["quantized_by_format"][fmt], X, reference_output)
        for fmt in FORMATS
    }
    reference_results: Dict[str, dict] = {}
    if run_reference:
        for fmt in FORMATS:
            Wq_ref, info = strong_reference_search(
                W, X, fmt, mode=reference_mode,
                ref_window=ref_window, ref_window_max=ref_window_max,
                ref_max_elems=ref_max_elems, n_random=n_random,
                pair_moves=pair_moves, seed=0, bfp_sharing=bfp_sharing,
            )
            reference_results[fmt] = {
                "W_quantized": Wq_ref,
                "output_error_pct": output_error_pct(W, Wq_ref, X, reference_output),
                "search_info": info,
            }
    return {
        "fast": fast,
        "fast_seconds": fast_seconds,
        "baseline_output_error_pct": baseline_errors,
        "reference": reference_results,
        "baseline_actual_winner": min(FORMATS, key=lambda fmt: baseline_errors[fmt]),
        "reference_winner": (
            min(FORMATS, key=lambda fmt: reference_results[fmt]["output_error_pct"])
            if reference_results else None
        ),
    }

def _print_table(results: dict) -> None:
    fast = results["fast"]
    print("\n=== FORMAT COMPARISON ===")
    has_reference = bool(results["reference"])
    headers = ["Format", "Fast sampled RMSE", "Baseline output error %"]
    if has_reference:
        headers += ["Reference output error %", "Reference search"]
    rows = []
    for fmt in FORMATS:
        row = [fmt, f"{fast['sampled_scores'][fmt]:.8g}", f"{results['baseline_output_error_pct'][fmt]:.8g}"]
        if has_reference:
            info = results["reference"][fmt]["search_info"]
            row += [f"{results['reference'][fmt]['output_error_pct']:.8g}", info["method"]]
        rows.append(row)
    widths = [max(len(headers[j]), *(len(row[j]) for row in rows)) for j in range(len(headers))]
    print(" | ".join(headers[j].ljust(widths[j]) for j in range(len(headers))))
    print("-+-".join("-" * width for width in widths))
    for row in rows:
        print(" | ".join(row[j].ljust(widths[j]) for j in range(len(headers))))

    print(f"\nFast selector chose: {fast['chosen']}")
    print(f"Fast selector runtime: {results['fast_seconds']:.4f} s")
    print(f"Best baseline format by full calibration error: {results['baseline_actual_winner']}")
    print(f"Selected X columns (0-based): {fast['selected_columns']}")
    if fast["full_tie_scores"]:
        print("Full-X scores used to resolve a near tie:", fast["full_tie_scores"])
    if has_reference:
        winner = results["reference_winner"]
        info = results["reference"][winner]["search_info"]
        print(f"Scale-optimized reference winner: {winner}")
        print(f"Reference winner output error: {results['reference'][winner]['output_error_pct']:.8g}%")
        print(f"Reference search runtime (winner only): {info.get('seconds', float('nan')):.4f} s")
        print("Reference method by format:", {fmt: results["reference"][fmt]["search_info"]["method"] for fmt in FORMATS})
        if any(results["reference"][fmt]["search_info"]["heuristic_rows"] for fmt in FORMATS):
            print("WARNING: at least one reference format used heuristic rows; those results are not proven globally optimal.")


## 2. Set paths and options

Set `WEIGHTS_PATH` and `CALIBRATION_PATH` to the input files. Leave `TEST_PATH = None` if no separate test inputs are available.

`SEED` controls the repeatable sample. `REFERENCE_MODE` chooses `auto`, `exact`, or `heuristic` scale search. The window and size settings control how much reference work is allowed. Set `RUN_REFERENCE = False` to skip the scale-search comparison.


In [2]:
from pathlib import Path

WEIGHTS_PATH = Path("c:/Users/Admin/OneDrive/Desktop/System/W_C.npy")
CALIBRATION_PATH = Path("c:/Users/Admin/OneDrive/Desktop/System/X_cal.npy")
TEST_PATH = None

SEED = 271828
BFP_SHARING = "block"

REFERENCE_MODE = "auto"
REF_WINDOW = 3
REF_WINDOW_MAX = 5
MAX_SEARCH_ELEMENTS = 2_000_000
RANDOM_STARTS = 4
PAIR_MOVES = True
RUN_REFERENCE = True


## 3. Load and check the data

Load the arrays and check that they are finite, nonempty matrices. The number of weight columns must equal the number of input rows. Each calibration-input column is one example. Optional test inputs must have the same number of rows.


In [3]:
if not WEIGHTS_PATH.exists():
    raise FileNotFoundError(
        f"Weights file not found: {WEIGHTS_PATH.resolve()}. "
        "Edit WEIGHTS_PATH in the configuration cell."
    )
if not CALIBRATION_PATH.exists():
    raise FileNotFoundError(
        f"Calibration file not found: {CALIBRATION_PATH.resolve()}. "
        "Edit CALIBRATION_PATH in the configuration cell."
    )

W = np.load(WEIGHTS_PATH, allow_pickle=False)
X_cal = np.load(CALIBRATION_PATH, allow_pickle=False)
W, X_cal = validate_matrices(W, X_cal, "X_cal")
X_test = None
if TEST_PATH is not None:
    if not Path(TEST_PATH).exists():
        raise FileNotFoundError(f"Test file not found: {Path(TEST_PATH).resolve()}")
    X_test = np.load(TEST_PATH, allow_pickle=False)
    _, X_test = validate_matrices(W, X_test, "X_test")

print("W shape:", W.shape)
print("X_cal shape:", X_cal.shape)
print("Calibration output shape:", (W @ X_cal).shape)
print("Numerical convention: 32-weight row-wise blocks; exponent [-127, 127]; round-to-nearest-even.")
print("MXUINT4: custom midpoint-coded convention, unsigned code 0..15 -> signed level code-8, fixed zero point 8.")
print(f"BFP4 sharing: {BFP_SHARING}; exponent policy: {BFP_EXP_POLICY}")
if X_test is not None:
    print("X_test shape:", X_test.shape, "(held out; not used for selection or scale tuning)")


W shape: (100, 100)
X_cal shape: (100, 256)
Calibration output shape: (100, 256)
Numerical convention: 32-weight row-wise blocks; exponent [-127, 127]; round-to-nearest-even.
MXUINT4: custom midpoint-coded convention, unsigned code 0..15 -> signed level code-8, fixed zero point 8.
BFP4 sharing: block; exponent policy: autoflex_predict


## 4. Run and compare

Run the sampled selector, measure every format's full-baseline output error, and optionally run the scale search.

The printed results compare selected formats, errors, time, and search status. Lower error is better. The selector's estimated RMSE and the full-output error percentage use different units, so compare choices and the full-error results rather than treating these numbers as interchangeable.


In [4]:
results = compare_algorithms(
    W, X_cal,
    seed=SEED,
    reference_mode=REFERENCE_MODE,
    ref_window=REF_WINDOW,
    ref_window_max=REF_WINDOW_MAX,
    ref_max_elems=MAX_SEARCH_ELEMENTS,
    n_random=RANDOM_STARTS,
    pair_moves=PAIR_MOVES,
    bfp_sharing=BFP_SHARING,
    run_reference=RUN_REFERENCE,
)
_print_table(results)



=== FORMAT COMPARISON ===
Format  | Fast sampled RMSE | Baseline output error % | Reference output error % | Reference search
--------+-------------------+-------------------------+--------------------------+-----------------
E2M1    | 0.12472344        | 12.552314               | 12.03066                 | EXACT           
E1M2    | 0.16328675        | 15.817232               | 14.004212                | EXACT           
MXUINT4 | 0.17385818        | 17.126238               | 13.45206                 | EXACT           
MXINT4  | 0.16312578        | 15.807957               | 13.45206                 | EXACT           
BFP4    | 0.16312578        | 15.807957               | 13.45206                 | EXACT           

Fast selector chose: E2M1
Fast selector runtime: 1.3232 s
Best baseline format by full calibration error: E2M1
Selected X columns (0-based): [114, 10, 213, 223, 129, 55, 172, 224, 51, 79, 75, 141, 137, 35, 42, 49, 126, 193, 30, 85, 201, 103, 186, 145, 169, 135, 86, 194, 1

## 5. Save the chosen weights

Save the fast selector's rounded weight matrix and, when available, the scale-search winner's matrix separately. These contain reconstructed floating-point values, not packed 4-bit storage.


In [5]:
FAST_OUTPUT_PATH = Path("W_quantized_fast.npy")
REFERENCE_OUTPUT_PATH = Path("W_quantized_reference.npy")

np.save(FAST_OUTPUT_PATH, results["fast"]["W_quantized"])
print(f"Saved fast-selector weights to: {FAST_OUTPUT_PATH.resolve()}")

if results["reference"]:
    reference_winner = results["reference_winner"]
    Wq_reference = results["reference"][reference_winner]["W_quantized"]
    np.save(REFERENCE_OUTPUT_PATH, Wq_reference)
    print(f"Reference winner: {reference_winner}")
    print(f"Saved reference weights to: {REFERENCE_OUTPUT_PATH.resolve()}")
else:
    Wq_reference = None
    print("Reference search was skipped; no reference output was saved.")


Saved fast-selector weights to: C:\Users\Admin\OneDrive\Desktop\System\Algo-4\W_quantized_fast.npy
Reference winner: E2M1
Saved reference weights to: C:\Users\Admin\OneDrive\Desktop\System\Algo-4\W_quantized_reference.npy


## 6. Check separate test inputs if available

After choosing formats using calibration data, measure their output errors on `X_test`. This checks how the choices perform on examples not used for selection. Test results do not take part in format selection or scale tuning.


In [6]:
if X_test is None:
    print("Held-out test evaluation skipped. Set TEST_PATH to your separate X_test.npy file and rerun the load, comparison, and evaluation cells.")
else:
    fast_test_error = output_error_pct(W, results["fast"]["W_quantized"], X_test)
    print(f"Fast-selector winner ({results['fast']['chosen']}) test output error: {fast_test_error:.8g}%")
    if results["reference"]:
        ref_winner = results["reference_winner"]
        ref_test_error = output_error_pct(W, results["reference"][ref_winner]["W_quantized"], X_test)
        print(f"Reference winner ({ref_winner}) test output error: {ref_test_error:.8g}%")


Held-out test evaluation skipped. Set TEST_PATH to your separate X_test.npy file and rerun the load, comparison, and evaluation cells.
